# term

> Terminals: gateway-hosted ptys, siblings of kernels

In [ ]:
#| default_exp term

Kernels moved behind the gateway, but the terminal features of jupygate's clients (solveit's embedded terminal, ipyai's shell mode) spawn ptys in their own process — the wrong side of the gateway the moment the kernel is remote. A terminal must live where the kernel lives: same machine, container, and filesystem. So terminals are a gateway resource, siblings of kernels, mirroring [jupyter's terminals API](https://github.com/jupyter-server/jupyter_server_terminals): named terminals you can list, create, delete, and attach to over a websocket — with replay on reattach, so a page refresh or dropped connection resumes the *same* shell with recent scrollback. The pty sessions themselves are [ptymini](https://github.com/AnswerDotAI/ptymini)'s (`PtySession`/`PtyRegistry`, extracted from here); this module owns what is genuinely the gateway's: the REST translation, the websocket framing, auth, and privilege policy (`_sudo`). Design notes and the choreography audit live in `meta/TERM.md`.

The websocket framing is bytes-native rather than [terminado](https://github.com/jupyter/terminado)'s JSON-of-decoded-text: binary frames carry pty bytes verbatim in both directions, and text frames carry JSON control messages. Client parsers that watch the stream for escape sequences (ipyai's sentinel choreography) need the raw bytes; decode/re-encode is a fidelity risk with nothing bought.


In [ ]:
#| export
import asyncio, json
from starlette.responses import JSONResponse, Response
from starlette.routing import Route, WebSocketRoute
from ptymini.core import PtyRegistry, Gap
from jupygate.core import _authed, _sudo


In [ ]:
import httpx, json, os, time
from jupygate.core import create_app, serve
from fastcore.test import test_eq
from websockets.sync.client import connect as ws_connect

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')


## The routes

The REST surface is jupyter's terminals API; the websocket is ours. `term_routes` builds the route list against one `PtyRegistry`, with the same `guard` auth/404 shape as the kernels routes, so `create_app` mounts terminals with two lines. `username` is handled here — privilege is gateway policy, so the route prefixes `_sudo` onto the argv rather than ptymini knowing about users. The ws contract:

- **binary frames**: pty bytes, verbatim, both directions
- **text frames**: JSON control — client sends `{"type": "set_size", "rows": R, "cols": C}`; server sends `{"type": "setup", "name": N}` on accept (replayed scrollback follows as binary), `{"type": "gap", "bytes": N}` when this client fell behind the replay ring and lost N bytes (clear-and-redraw territory), and `{"type": "eof", "code": C}` when the pty dies

Connecting to a name that doesn't exist closes with 4404 rather than auto-creating: creation carries options (argv, rc, env), so it belongs to POST, and a typo'd reattach should fail loudly, not spawn a stray shell.


In [ ]:
#| export
def term_routes(terminals:PtyRegistry, auth_token:str|None=None)->list:
    "Starlette routes for the terminals API, against one ptymini `PtyRegistry`."
    async def list_terms(request): return JSONResponse([t.model() for t in terminals.values()])

    async def create_term(request):
        body = await request.json() if await request.body() else {}
        kw = {k: body[k] for k in ('name','argv','cwd','env','appendenv','rc','rows','cols') if k in body}
        if body.get('username'): kw['argv'] = [*_sudo(body['username']), *(kw.get('argv') or terminals.argv)]
        try: t = await terminals.create(**kw)
        except Exception as e: return JSONResponse(dict(message=f'terminal failed to start: {e}'), status_code=500)
        return JSONResponse(t.model(), status_code=201)

    async def get_term(request): return JSONResponse(terminals.terms[request.path_params['name']].model())

    async def delete_term(request):
        await terminals.delete(request.path_params['name'])
        return Response(status_code=204)

    async def channel(ws):
        if not _authed(ws, auth_token): return await ws.close(code=4403)
        t = terminals.get(ws.path_params['name'])
        if t is None: return await ws.close(code=4404)
        await ws.accept()
        await ws.send_text(json.dumps(dict(type='setup', name=t.name)))
        async def pump():
            async for item in t.attach(gaps=True):
                if isinstance(item, Gap): await ws.send_text(json.dumps(dict(type='gap', bytes=int(item))))
                else: await ws.send_bytes(item)
            await ws.send_text(json.dumps(dict(type='eof', code=t.exit_code)))
        task = asyncio.create_task(pump())
        try:
            while True:
                event = await ws.receive()
                if event['type'] == 'websocket.disconnect': break
                if event.get('bytes'): t.write(event['bytes'])
                elif event.get('text'):
                    c = json.loads(event['text'])
                    if c.get('type') == 'set_size': t.resize(c['rows'], c['cols'])
        finally: task.cancel()

    def guard(fn):
        async def inner(request):
            if not _authed(request, auth_token): return JSONResponse(dict(message='forbidden'), status_code=403)
            try: return await fn(request)
            except KeyError: return JSONResponse(dict(message='no such terminal'), status_code=404)
        return inner

    r = lambda p,meth,f: Route('/api/terminals'+p, guard(f), methods=[meth])
    return [r('','GET',list_terms), r('','POST',create_term), r('/{name}','GET',get_term),
        r('/{name}','DELETE',delete_term), WebSocketRoute('/api/terminals/{name}/channel', channel)]

## A live gateway

The gateway app mounts these routes next to the kernels API (`create_app` in `core`), so one server, one port, and one token cover both resources. The REST surface in the order a client uses it: nothing running, create one (this spawns a real shell), see it listed.

In [ ]:
server = serve(create_app(), port=0, in_thread=True)
http = httpx.Client(base_url=server.url, timeout=30)
test_eq(http.get('/api/terminals').json(), [])
model = http.post('/api/terminals', json=dict(argv=BASH, env=BENV)).json()
test_eq(http.get('/api/terminals').json()[0]['name'], model['name'])
model

Attach over the websocket. The first frame is the `setup` text frame; from there, binary out is keystrokes and binary in is the pty. `ws_until` accumulates binary frames until a pattern appears — the sync-websockets twin of `read_until`.

In [ ]:
def ws_until(ws, pat:bytes, timeout=10.0)->bytes:
    "Accumulate binary frames from sync websocket `ws` until `pat` appears (text frames are skipped)."
    buf = b''
    end = time.monotonic() + timeout
    while pat not in buf:
        frame = ws.recv(timeout=end - time.monotonic())
        if isinstance(frame, bytes): buf += frame
    return buf

wsurl = f"{server.url.replace('http', 'ws')}/api/terminals/{model['name']}/channel"
ws = ws_connect(wsurl)
setup = json.loads(ws.recv(timeout=10))
test_eq(setup['type'], 'setup')
ws.send(b'echo wired $((2*3))\n')
out = ws_until(ws, b'wired 6')
out[-20:]

The reattach story, end to end over the wire: a second connection to the same name gets `setup` and then the replayed scrollback — the output of a command that ran before this client existed. This is what a browser refresh sees.

In [ ]:
ws2 = ws_connect(wsurl)
json.loads(ws2.recv(timeout=10))['type'], b'wired 6' in ws_until(ws2, b'wired 6')

Control rides text frames: resize and check in-band, then DELETE the terminal and watch the open websockets learn of the death via `eof` (the pump sends it when the pty EOFs, so every attached client hears, not just the one that asked).

In [ ]:
ws.send(json.dumps(dict(type='set_size', rows=50, cols=120)))
ws.send(b'stty size\n')
assert b'50 120' in ws_until(ws, b'50 120')
test_eq(http.delete(f"/api/terminals/{model['name']}").status_code, 204)
frame = ws.recv(timeout=10)
while isinstance(frame, bytes): frame = ws.recv(timeout=10)   # drain any final output; the text frame is the eof
test_eq(json.loads(frame)['type'], 'eof')
test_eq(http.get('/api/terminals').json(), [])

In [ ]:
#| hide
ws.close()
ws2.close()
http.close()
server.should_exit = True


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()